In [1]:
import fitz as pymupdf
import os
import re
import unicodedata


DEVANAGARI_RE = re.compile(r'[\u0900-\u097F]')
LATIN_RE = re.compile(r'[a-zA-Z]')


# accepts:
# 1. Title
# १. Title
TITLE_RE = re.compile(
    r'^(?:[0-9]+|[०-९]+)[\.\)]\s*(.+)$'
)


def clean(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u200c", "").replace("\u200d", "")
    return re.sub(r'\s+', ' ', text).strip()



def detect_language(text):

    devanagari = len(DEVANAGARI_RE.findall(text))
    latin = len(LATIN_RE.findall(text))

    if devanagari > latin:
        return "sanskrit"

    if latin > devanagari:
        return "english"

    return "unknown"



def extract_title(line):

    match = TITLE_RE.match(line)

    if not match:
        return None


    title = match.group(1).strip()


    # ignore long numbered paragraphs
    if len(title.split()) > 10:
        return None


    # ignore sentence-like lines
    if title.endswith((".", "।", ":", ",")):
        return None


    return title




def extract_bilingual(file_path):

    doc = pymupdf.open(file_path)

    stories = []

    current = {
        "title": None,
        "english": [],
        "sanskrit": []
    }

    waiting_for_content = False



    def save_story():

        if current["english"] or current["sanskrit"]:

            stories.append(
                {
                    "title": current["title"],
                    "english": " ".join(current["english"]),
                    "sanskrit": " ".join(current["sanskrit"])
                }
            )



    for page in doc:


        lines = [
            clean(x)
            for x in page.get_text("text").splitlines()
            if clean(x)
        ]


        for line in lines:


            if line == "Automatic Zoom":
                continue



            title = extract_title(line)



            # ==============================
            # TITLE FOUND
            # ==============================

            if title:


                # first title
                if current["title"] is None:

                    current["title"] = title
                    waiting_for_content = True
                    continue



                # second title of same story
                #
                # Example:
                #
                # 1. सिंहः च मूषकः
                # Sanskrit text
                # 1. The Lion and the Mouse
                #
                # This will be handled later by merge
                #
                if (
                    waiting_for_content
                    and not current["english"]
                    and not current["sanskrit"]
                ):
                    continue



                # new story
                save_story()


                current = {
                    "title": title,
                    "english": [],
                    "sanskrit": []
                }


                waiting_for_content = True

                continue




            # ==============================
            # CONTENT
            # ==============================

            lang = detect_language(line)



            if lang == "english":

                current["english"].append(line)
                waiting_for_content = False



            elif lang == "sanskrit":

                current["sanskrit"].append(line)
                waiting_for_content = False




    # save last story
    save_story()


    return stories


def merge_bilingual_stories(stories):

    merged = []

    i = 0

    while i < len(stories):

        current = stories[i]


        if i + 1 < len(stories):

            nxt = stories[i + 1]


            # Sanskrit followed by English
            if (
                current["sanskrit"]
                and not current["english"]
                and nxt["english"]
                and not nxt["sanskrit"]
            ):

                merged.append(
                    {
                        "title": {
                            "sanskrit": current["title"],
                            "english": nxt["title"]
                        },
                        "english": nxt["english"],
                        "sanskrit": current["sanskrit"]
                    }
                )

                i += 2
                continue



            # English followed by Sanskrit
            if (
                current["english"]
                and not current["sanskrit"]
                and nxt["sanskrit"]
                and not nxt["english"]
            ):

                merged.append(
                    {
                        "title": {
                            "english": current["title"],
                            "sanskrit": nxt["title"]
                        },
                        "english": current["english"],
                        "sanskrit": nxt["sanskrit"]
                    }
                )

                i += 2
                continue


        merged.append(current)

        i += 1


    return merged



# =========================================================
# RUN
# =========================================================

file_path = os.path.join(
    "..",
    "data",
    "sanskrit_english_bilingual_stories.pdf"
)


data = extract_bilingual(file_path)
data = merge_bilingual_stories(data)



for i, story in enumerate(data, 1):
    print("\nStory", i)
    print("----------------")

    print("TITLE:", story["title"])

    print("\nEnglish:")
    print(
        story["english"]
    )

    print("\nSanskrit:")
    print(
        story["sanskrit"]
    )


Story 1
----------------
TITLE: {'sanskrit': 'ɭसिंहः च मूषकः', 'english': 'The Lion and the Mouse'}

English:
In a forest, there lived a powerful lion. One day, he was sleeping under a tree. Just then, a small mouse came there and started running up and down the lion's body. Because of this, the lion's sleep was disturbed. The lion became angry and caught the mouse in his paw. The mouse, terrified, said, 'O King of the Forest! Please spare me. Someday I too may help you.' The lion laughed and let him go. After some time, the lion was caught in a hunter's net. He roared loudly. Hearing the lion's roar, the mouse arrived there. He gnawed the net with his sharp teeth. The lion became free from the net. He thanked the mouse.

Sanskrit:
एकɧ×स्मिन् वने एकः बलवान् ɭसिंहः वसिɟति ×स्मि। एकदा सिः वृक्ष×य अधः सिुप्तिः आसिीति्। तिदा एकः लघुः स्मिूषकः तित्र आगत्य ɭसिंह×य शरीरे इति×तितिः अधावति्। तिेन ɭसिंह×य ɟनद्रा भग्ना अभवति्। ɭसिंहः क्रुद्धः भूत्वा स्मिूषकं ह×तिेन अगृह्णाति्। स्मिूषकः भीतिः भूत